#Tweet extraction with date and hours

Extraction part

In [ ]:
# --- Config ---
DATE_LIMIT = datetime(2023, 11, 5)
BASE_URL = "https://rollcall.com/wp-json/factbase/v1/twitter"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": "https://rollcall.com/factbase/trump/topic/social/",
    "Accept": "application/json"
}

results = []
page = 1
stop = False

while not stop:
    params = {
        "platform": "all",
        "sort": "date",
        "sort_order": "desc",
        "page": page,
        "format": "json"
    }

    try:
        resp = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print(f"❌ Erreur page {page} : {e}")
        break

    # L'API retourne soit une liste, soit un dict avec une clé
    items = data if isinstance(data, list) else data.get("data", data.get("results", data.get("posts", [])))

    if not items:
        print(f"✅ Plus de données à la page {page} → arrêt.")
        break

    print(f"  Page {page} — {len(items)} tweets reçus", end="")

    for item in items:
        # --- Date ---
        raw_date = item.get("date", "")
        try:
            tweet_datetime = datetime.fromisoformat(raw_date.replace("Z", "+00:00"))
            tweet_date = tweet_datetime.date()
            tweet_time = tweet_datetime.strftime("%H:%M:%S")
        except:
            continue

        if tweet_date < DATE_LIMIT.date():
            print(f"\n📅 Date {tweet_date} < 2023-11-05 → arrêt !")
            stop = True
            break

        # --- Texte : extraire les <p> du post_html ---
        post_html = item.get("social", {}).get("post_html", "") or item.get("post_html", "")
        texts = []
        if post_html:
            soup = BeautifulSoup(post_html, "html.parser")
            for p in soup.find_all("p"):
                t = p.get_text(strip=True)
                if t:
                    texts.append(t)

        # --- URL ---
        url = item.get("social", {}).get("url", "") or item.get("url", "")

        if texts or url:
            results.append({
                "date": tweet_date.strftime("%Y-%m-%d"),
                "time_et": tweet_time,
                "datetime_full": raw_date,
                "url": url,
                "text": " ".join(texts)
            })
    print(f" — {len(results)} résultats cumulés")
    page += 1
    time.sleep(0.5)  # Pause légère pour ne pas surcharger le serveur

# --- Affichage ---
print(f"\n📊 Total tweets extraits (≥ 2023-11-05) : {len(results)}\n")
for i, item in enumerate(results, 1):
    print(f"[{item['date']} {item['time_et']}]")
    if item['url']:
        print(f"  🔗 {item['url']}")
    if item['text']:
        print(f"  💬 {item['text']}")
    print()

# --- Sauvegarde JSON optionnelle ---
with open("trump_tweets.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("💾 Sauvegardé dans trump_tweets.json")

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.

[2024-01-05 19:23:42]
  💬 RT@RSBN“Our Founding Fathers would be turning in their graves if they could see this.” — President Trump on Democrats’ political weaponization of law enforcement in Sioux Center, Iowa Watch LIVE:https://www.rsbnetwork.com/video/live-trump-to-speak-at-ia-commit-to-caucus-rallies-in-sioux-center-and-mason-city-1-5-24/

[2024-01-05 19:23:35]
  💬 RT@RSBN“We’ll immediately restore and expand the Trump travel ban for those countries that are terror-plagued.” — President Trump in Sioux Center, Iowa Watch LIVE:https://www.rsbnetwork.com/video/live-trump-to-speak-at-ia-commit-to-caucus-rallies-in-sioux-center-and-mason-city-1-5-24/

[2024-01-05 19:22:36]
  💬 RT@realamericasvoiceWe gotta get out and vote because you know bad things happen... (when you do nothing) - President@realDonaldTrump Join our LIVE Coverage of President Trump’s Speech:https://rumble.com/v45bbf5-president-trump-commit-to

In [ ]:
# Load tweets from Drive
base_path = "/content/drive/MyDrive/M2Economiedudévleoppement/Text analysis/Poster"

with open("trump_tweets.json", "r", encoding="utf-8") as f:
    tweets = json.load(f)

# Afficher le nombre total de tweets
print(f"📊 Nombre total de tweets : {len(tweets)}")

# Afficher les dates du premier et du dernier tweet
if tweets:
    dates = [tweet["date"] for tweet in tweets]
    print(f"📅 Premier tweet : {dates[-1]}")
    print(f"📅 Dernier tweet : {dates[0]}")

# Vérifier si la date limite est respectée
    date_limite = datetime(2023, 11, 5).date()
    dernier_tweet_date = datetime.strptime(dates[-1], "%Y-%m-%d").date()
    if dernier_tweet_date >= date_limite:
        print("✅ Tous les tweets jusqu'à la date limite sont présents.")
    else:
        print("⚠️ Il manque des tweets après la date limite.")
else:
    print("❌ Aucun tweet trouvé dans le fichier.")

# Créer le DataFrame
df = pd.DataFrame(tweets)
print(f"📊 DataFrame shape: {df.shape}")
print(df[["date", "time_et", "text"]].head())


📊 Nombre total de tweets : 16492
📅 Premier tweet : 2023-11-05
📅 Dernier tweet : 2026-03-16
✅ Tous les tweets jusqu'à la date limite sont présents.
📊 DataFrame shape: (16492, 5)
         date   time_et                                               text
0  2026-03-16  11:57:47  Susie Wiles is an incredible Chief of Staff, a...
1  2026-03-16  09:19:11  News Conference today, prior to Trump Kennedy ...
2  2026-03-16  09:16:38  The crazed Democrats are not allowing TSA Agen...
3  2026-03-16  07:50:09  Michael Whatley is so Great! Running against a...
4  2026-03-15  22:46:08  RT: https://truthsocial.com/users/realDonaldTr...


The tweets are already saved

In [ ]:
!pip install nltk pandas
import nltk
nltk.download('punkt')       # Pour la tokenisation
nltk.download('wordnet')    # Pour la lemmatisation
nltk.download('stopwords')  # Pour les stopwords

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

#BERT Embeddings

In [ ]:
# Définir le device (GPU si disponible, sinon CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du device : {device}")

# Charger le tokeniseur et le modèle BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
bert_model.to(device)
bert_model.eval()

# Préparer les textes pour BERT
texts = df["text"].tolist()
batch_size = 32
embeddings = []

# Calculer les embeddings
with torch.no_grad():
    for i in tqdm(range(0, len(texts), batch_size), desc="Calcul des embeddings"):
        batch_texts = texts[i:i+batch_size]
        # Tokenisation par batch — padding uniquement au sein du batch
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = bert_model(**inputs)
        embeddings.extend(outputs.pooler_output.cpu())


# Stack les embeddings dans un tenseur
embeddings = torch.stack(embeddings)
print(f"Forme des embeddings : {embeddings.shape}")


Utilisation du device : cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Calcul des embeddings: 100%|██████████| 516/516 [04:19<00:00,  1.98it/s]


Forme des embeddings : torch.Size([16492, 768])


Sauvegarde sur drive

In [ ]:
drive.mount('/content/drive')

# Chemin vers votre dossier
base_path = "/content/drive/MyDrive/M2Economiedudévleoppement/Text analysis/Poster"

# Créer le dossier s'il n'existe pas
os.makedirs(base_path, exist_ok=True)

# Chemin pour sauvegarder les tweets
save_path_tweets = os.path.join(base_path, "trump_tweets.json")

# Sauvegarder les tweets
with open(save_path_tweets, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Tweets sauvegardés dans : {save_path_tweets}")

# Chemin pour sauvegarder les embeddings
save_path_embeddings = os.path.join(base_path, "trump_tweets_embeddings.pt")

# Sauvegarder les embeddings
torch.save(embeddings, save_path_embeddings)

print(f"Embeddings sauvegardés dans : {save_path_embeddings}")



Mounted at /content/drive
Tweets sauvegardés dans : /content/drive/MyDrive/M2Economiedudévleoppement/Text analysis/Poster/trump_tweets.json
Embeddings sauvegardés dans : /content/drive/MyDrive/M2Economiedudévleoppement/Text analysis/Poster/trump_tweets_embeddings.pt


Structuratio  des tweets :
date : Date du tweet (ex: 2026-03-11).
time_et : Heure du tweet (ex: 19:39:01).
datetime_full : Horodatage complet du tweet (ex: 2026-03-11T19:39:01-04:00).
url : URL du tweet.
text : Texte du tweet.
